# ⌚ Apple Watch Data Lake - Exploratory Data Analysis (EDA)

Este notebook permite explorar de forma interativa e iterativa a camada **Silver** (arquivos Parquet estruturados) do nosso Data Lake local hospedado no **MinIO**, utilizando a engine SQL do **DuckDB**.

### 1. Configurando Conexões e Dependências

In [1]:
import sys
import duckdb
from pathlib import Path

# Adiciona o diretório raiz ao path do Python para poder importar do src
project_root = str(Path.cwd().resolve())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.config import settings
from src.utils.minio_client import get_minio_client

# Inicializa conexão local do DuckDB
conn = duckdb.connect()

# Instala e carrega a extensão httpfs para suportar conexões S3 (MinIO)
conn.execute("INSTALL httpfs;")
conn.execute("LOAD httpfs;")

# Configura as credenciais locais do MinIO no DuckDB
conn.execute(f"SET s3_endpoint='{settings.minio_endpoint}';")
conn.execute(f"SET s3_access_key_id='{settings.minio_access_key}';")
conn.execute(f"SET s3_secret_access_key='{settings.minio_secret_key}';")
conn.execute("SET s3_use_ssl=false;")
conn.execute("SET s3_url_style='path';")

print("✅ DuckDB configurado com sucesso para consultar o MinIO!")

✅ DuckDB configurado com sucesso para consultar o MinIO!


### 2. Escaneando Partições Disponíveis na Camada Silver

In [2]:
# Lista dinamicamente as pastas presentes no bucket Silver do MinIO
client = get_minio_client()
objects = client.list_objects(settings.bucket_silver, recursive=False)

print("📂 Partições estruturadas prontas para consulta na Silver:")
for obj in objects:
    if obj.is_dir:
        dir_name = obj.object_name.strip("/")
        print(f"  - {dir_name}")

📂 Partições estruturadas prontas para consulta na Silver:
  - records_type=active_energy
  - records_type=basal_energy
  - records_type=blood_oxygen
  - records_type=body_mass
  - records_type=breathing_disturbances
  - records_type=distance
  - records_type=exercise_time
  - records_type=flights_climbed
  - records_type=heart_rate
  - records_type=hrv
  - records_type=physical_effort
  - records_type=respiratory_rate
  - records_type=resting_heart_rate
  - records_type=sleep
  - records_type=stand_hour
  - records_type=stand_time
  - records_type=step_count
  - records_type=time_in_daylight
  - records_type=vo2_max
  - records_type=walking_speed
  - records_type=wrist_temperature
  - workouts


### 3. Exemplos de Consultas Analíticas

#### A. Visualizando Amostra de Batimentos Cardíacos (`heart_rate`)

In [4]:
query_hr = f"""
    SELECT * 
    FROM read_parquet('s3://{settings.bucket_silver}/records_type=heart_rate/*.parquet') 
    LIMIT 10
"""
conn.execute(query_hr).df()

,source_name,source_version,device,unit,creation_date,start_date,end_date,value,type,records_type
0,Rafael’s Apple Watch,31.2,"<<HKDevice: 0xcdbf45b60>, name:Apple Watch, ma...",count/min,2026-04-01 07:37:14,2026-04-01 07:31:31,2026-04-01 07:31:31,94.0,heart_rate,heart_rate
1,Rafael’s Apple Watch,31.2,"<<HKDevice: 0xcdbf45b60>, name:Apple Watch, ma...",count/min,2026-04-01 07:52:45,2026-04-01 07:37:12,2026-04-01 07:37:12,82.0,heart_rate,heart_rate
2,Rafael Panegassi,26.4,"<<HKDevice: 0xcdbf45b60>, name:Apple Watch, ma...",count/min,2026-04-01 07:17:42,2026-04-01 07:17:42,2026-04-01 07:17:42,58.0,heart_rate,heart_rate
3,Rafael Panegassi,26.4,"<<HKDevice: 0xcdbf45b60>, name:Apple Watch, ma...",count/min,2026-04-01 06:47:42,2026-04-01 06:47:42,2026-04-01 06:47:42,50.0,heart_rate,heart_rate
4,Rafael Panegassi,26.4,"<<HKDevice: 0xcdbf45b60>, name:Apple Watch, ma...",count/min,2026-04-01 06:16:51,2026-04-01 06:16:51,2026-04-01 06:16:51,48.0,heart_rate,heart_rate
5,Rafael Panegassi,26.4,"<<HKDevice: 0xcdbf45b60>, name:Apple Watch, ma...",count/min,2026-04-01 05:35:00,2026-04-01 05:35:00,2026-04-01 05:35:00,48.0,heart_rate,heart_rate
6,Rafael Panegassi,26.4,"<<HKDevice: 0xcdbf45b60>, name:Apple Watch, ma...",count/min,2026-04-01 04:49:10,2026-04-01 04:49:10,2026-04-01 04:49:10,48.0,heart_rate,heart_rate
7,Rafael Panegassi,26.4,"<<HKDevice: 0xcdbf45b60>, name:Apple Watch, ma...",count/min,2026-04-01 04:18:44,2026-04-01 04:18:44,2026-04-01 04:18:44,46.0,heart_rate,heart_rate
8,Rafael Panegassi,26.4,"<<HKDevice: 0xcdbf45b60>, name:Apple Watch, ma...",count/min,2026-04-01 03:36:23,2026-04-01 03:36:23,2026-04-01 03:36:23,47.0,heart_rate,heart_rate
9,Rafael Panegassi,26.4,"<<HKDevice: 0xcdbf45b60>, name:Apple Watch, ma...",count/min,2026-04-01 03:06:23,2026-04-01 03:06:23,2026-04-01 03:06:23,49.0,heart_rate,heart_rate


#### B. Consultando Métricas Novas (Ex: Saturação de Oxigênio - `blood_oxygen`)

In [5]:
query_oxygen = f"""
    SELECT 
        CAST(start_date AS DATE) as data,
        ROUND(AVG(value) * 100, 2) as avg_oxygen_percentage,
        COUNT(*) as leituras
    FROM read_parquet('s3://{settings.bucket_silver}/records_type=blood_oxygen/*.parquet')
    GROUP BY 1
    ORDER BY 1 DESC
    LIMIT 10
"""
conn.execute(query_oxygen).df()

,data,avg_oxygen_percentage,leituras
0,2026-06-04,97.33,18
1,2026-06-03,97.75,20
2,2026-06-02,97.22,18
3,2026-06-01,97.62,24
4,2026-05-31,96.43,28
5,2026-05-30,97.29,17
6,2026-05-29,97.28,18
7,2026-05-28,96.39,23
8,2026-05-27,96.33,18
9,2026-05-26,97.20,20


#### C. Consultando Frequência Respiratória (`respiratory_rate`)

In [6]:
query_resp = f"""
    SELECT 
        MIN(value) as min_respiratory_rate,
        MAX(value) as max_respiratory_rate,
        ROUND(AVG(value), 1) as avg_respiratory_rate
    FROM read_parquet('s3://{settings.bucket_silver}/records_type=respiratory_rate/*.parquet')
"""
conn.execute(query_resp).df()

,min_respiratory_rate,max_respiratory_rate,avg_respiratory_rate
0,12.5,25.5,17.5


#### D. Consultando Resumo de Treinos (`workouts`)

In [7]:
query_workouts = f"""
    SELECT 
        workout_type, 
        COUNT(*) as total_treinos,
        ROUND(AVG(duration), 1) as duracao_media_min,
        ROUND(SUM(total_energy_burned), 0) as calorias_totais_burned_kcal
    FROM read_parquet('s3://{settings.bucket_silver}/workouts/*.parquet')
    GROUP BY 1
    ORDER BY 2 DESC
"""
conn.execute(query_workouts).df()

,workout_type,total_treinos,duracao_media_min,calorias_totais_burned_kcal
0,HKWorkoutActivityTypeWalking,26,45.8,0.0
1,HKWorkoutActivityTypeRunning,3,143.9,0.0
2,HKWorkoutActivityTypeCycling,1,5.0,0.0
3,HKWorkoutActivityTypeUnderwaterDiving,1,62.9,0.0


### 4. Escreva Suas Próprias Consultas SQL Abaixo!

In [12]:
# Altere a query abaixo à vontade e execute a célula
query_custom = f"""
    SELECT 
        type,
        COUNT(*) as total_records
    FROM read_parquet('s3://{settings.bucket_silver}/records_type=*/*.parquet')
    GROUP BY 1
    ORDER BY 2 DESC
"""
conn.execute(query_custom).df()

,type,total_records
0,heart_rate,185434
1,active_energy,46701
2,basal_energy,36759
3,distance,31635
4,physical_effort,26941
5,step_count,25970
6,walking_speed,11449
7,stand_time,4791
8,respiratory_rate,3402
9,sleep,2882
